# ICT-51 -- Origami Wolfram pli 4 : les combinateurs S/K/I, ou la representation ultime du calcul

**Navigation** : [Index](README.md) | Precedent dans la serie : [ICT-50 -- la regle 30 comme generateur d'alea, la regle 110 et son universalite](ICT-50-WolframRule30Aleatoire-Rule110Universelle-Python.ipynb)

**Carnet** : deux points de la definition du pli 4 qu'aucun carnet de la serie ne couvre encore --
la **logique combinatoire S/K/I** (syntaxe, reduction, forme normale, abstraction de bracket,
entiers de Church) et le **probleme de tag de Post** comme borne d'irreductibilite du geste
symbolique, avec, a cote de chaque these, sa **reception critique**.

**Organe utilise** : `ict.combinators` (chemin
`MyIA.AI.Notebooks/IIT/ICT-Series/ict/combinators.py`, livre par la meme PR que ce carnet --
meme forme que `ict.wolfram_step` du pli 2). Toutes les reductions et evolutions de ce carnet
passent par `combinators` ; **aucune reimplementation locale du moteur**. Ce que le carnet ecrit
lui-meme, ce sont les **instruments de presentation** (tables de traces, comparaisons de tailles),
qui ne sont pas dans l'organe.

**Ce que ce carnet fait, et ce qu'il ne fait pas.**

| Ces affirmations | Statut | Ce que fait ce carnet |
|---|---|---|
| S et K suffisent a representer tout calcul | *theoreme* (Schonfinkel 1924 ; Church 1936) | il **demonstre le mecanisme sur des exemples** (derivation de I, entiers de Church, arithmetique) ; il ne re-prouve pas l'universalite |
| La reduction est confluente : toute forme normale est unique (Church-Rosser) | *theoreme* | il en donne le **temoin empirique** (ordres de reduction aleatoires -> une seule forme normale) ; la preuve formelle est un grain separe de l'EPIC (organe Lean) |
| Le systeme de tag de Post (0->00, 1->1101, suppression de 3) s'arrete-t-il pour toute graine ? | *probleme ouvert* depuis 1921 | il **mesure** les comportements (arret, cycle, croissance) et les blocs inatteignables ; il ne tranche pas |

La distinction est le sujet d'une partie de ce carnet : un theoreme demontre sur exemples, un
temoin empirique et un probleme ouvert ne se citent pas de la meme facon.

> **Statut épistémique** — **Sans verdict à ce jour** : ce carnet n'a pas encore de ligne dans la [matrice de dissociations](../../../docs/ict/dissociations-matrix.md), donc aucune solidité n'est revendiquée ici. Ce qu'il mesure est reproductible depuis ses cellules ; ce n'est pas un acquis de la série.

## Sources primaires et regle de citation

Les textes utilises sont lus dans les copies archivees du gisement
`G:\Mon Drive\MyIA\IA\Bibliographie IA\` :

| Reference | Copie archivee |
|---|---|
| Wolfram, *Combinators: A Centennial View* (2020) | `Symbolic\Wolfram on Mathematics and Proof\2020 - Wolfram - Combinators - A Centennial View.html` |
| Wolfram, *After 100 Years, Can We Finally Crack Post's Problem of Tag?* (4 mars 2021) | `Automata\Wolfram Physics Project\2021 - Wolfram - After 100 Years, Can We Finally Crack Post's Problem of Tag.html` |
| Wolfram, *1920, 2020 and a $20,000 Prize: Announcing the S Combinator Challenge* (8 juin 2021) | `Symbolic\Wolfram on Mathematics and Proof\2021 - Wolfram - 1920-2020 and a $20000 Prize, Announcing the S Combinator Challenge.html` |
| Wolfram, *A Bibliography of Combinators* (arXiv 2106.11729, 2021) | `Symbolic\Wolfram on Mathematics and Proof\2021 - Wolfram - A Bibliography of Combinators (arXiv 2106.11729).pdf` |

**Convention de citation** : chaque affirmation attribuee a un texte cite la **section de la copie
archivee** (une capture HTML se cite par son titre de section, jamais par un numero de ligne ; le
PDF bibliographique par sa page). La reception critique (Curry, Hindley, Cardone, Bimbo, Klop)
est relevee dans la bibliographie archivee ci-dessus, pas dans les originaux -- et le carnet le
dit quand c'est le cas.

In [1]:
import random
import sys

sys.path.insert(0, 'MyIA.AI.Notebooks/IIT/ICT-Series')
from ict import combinators as cb

print('ict.combinators importe OK -- aucune reimplementation locale du moteur')
print('Regles : S f g x = f x (g x) | K x y = x | I x = x')

ict.combinators importe OK -- aucune reimplementation locale du moteur
Regles : S f g x = f x (g x) | K x y = x | I x = x


## 1. Trois lettres : la syntaxe minimale du calcul

Le 7 decembre 1920, Moses Schonfinkel presente a Gottingen les combinateurs ; l'article de 11
pages qui suit (*Uber die Bausteine der mathematischen Logik*, 1924) introduit deux regles et
une seule operation -- l'application. Wolfram resume ce geste : parvenir a concevoir "that it's
even conceivable that everything can be defined in terms of them is a **supreme achievement of
a kind of abstract reductionism**" (*A Centennial View*, section *Combinators Arrive*).

Toute la syntaxe tient dans ce tableau :

| Combinateur | Regle de reduction | Lecture |
|---|---|---|
| `S f g x` | `-> f x (g x)` | distribue l'argument aux deux branches |
| `K x y` | `-> x` | selectionne et oublie |
| `I x` | `-> x` | identite -- **derivable** : `S K K x -> x` |

La cellule suivante reduit `S K K x` pas a pas avec l'organe : le combinateur I n'est pas un
primitif, c'est un theoreme de deux pas.

In [2]:
term = cb.parse_ski('S K K x')
nf, steps, trace = cb.reduce_term(term, 'normal', 100, trace=True)
for i, t in enumerate(trace):
    print(f'pas {i}: {cb.to_str(t)}')
print('forme normale :', cb.to_str(nf), '| pas :', steps)

pas 0: S K K x
pas 1: K x (K x)
pas 2: x
forme normale : x | pas : 2


**Lecture du resultat** : `S K K x -> K x (K x) -> x`. Deux lettres suffisent ; la troisieme
(I) s'obtient. C'est ce constat qui fonde le **S Combinator Challenge** (8 juin 2021, prix de
$20 000) : Wolfram y demande si **S seul** -- sans K -- est encore universel, decrivant Schonfinkel
comme ayant fourni "the first explicit example of a system capable of what we now call universal
computation" (section *Hiding in Plain Sight for a Century?*). La question du minimum absolu
reste ouverte un siecle apres : c'est le fil rouge de ce pli.

## 2. Ordre normal contre ordre applicatif : la strategie n'est pas cosmetique

Deux strategies classiques de reduction :

- **ordre normal** : reduire le redex le plus externe-gauche d'abord -- le theoreme de
  standardisation garantit qu'il trouve la forme normale quand elle existe ;
- **ordre applicatif** : evaluer les arguments d'abord -- la strategie des langages usuels, qui
  peut diverger sur un terme qui possede pourtant une forme normale.

Le temoin canonique est `K I OMEGA` (ou `OMEGA = S I I (S I I)` est le terme qui se reduit en
lui-meme, section 3) : la regle K n'a jamais besoin de son second argument.

In [3]:
term = cb.App(cb.App(cb.K, cb.I), cb.OMEGA)
nf_n, s_n, _ = cb.reduce_term(term, 'normal', 500)
nf_a, s_a, _ = cb.reduce_term(term, 'applicative', 500)
print('ordre normal    :', cb.to_str(nf_n) if nf_n else 'diverge', f'({s_n} pas)')
print('ordre applicatif:', cb.to_str(nf_a) if nf_a else 'diverge', f'(borne {s_a} pas atteinte)')

ordre normal    : I (1 pas)
ordre applicatif: diverge (borne 500 pas atteinte)


**Lecture du resultat** : le **meme terme**, la **meme regle K** -- et deux destins. En ordre
normal, `K I OMEGA -> I` en un pas : l'argument jamais utilise n'est jamais evalue. En ordre
applicatif, la machine s'enfonce dans OMEGA et ne revient pas. La paresse n'est pas une
optimisation : c'est une difference **semantique** -- certains calculs n'existent que si l'on
s'interdit d'en faire certains autres.

## 3. OMEGA : la divergence pure

Le terme `S I I (S I I)` est le point fixe de la reduction : chaque pas produit un terme qui se
reduit encore. Wolfram note que, contrairement aux automates cellulaires ou aux machines de
Turing, les combinateurs ont "a certain immediate structural complexity, with their elaborate
tree structures" (*A Centennial View*, section *What Is Their Mathematics?*) -- la divergence
elle-meme y est structurale, pas un simple non-terminus.

In [4]:
print('terme initial :', cb.to_str(cb.OMEGA))
nf, steps, trace = cb.reduce_term(cb.OMEGA, 'normal', 3, trace=True)
for i, t in enumerate(trace):
    print(f'pas {i}: {cb.to_str(t)}')
verdict = cb.reduce_term(cb.OMEGA, 'normal', 200)[0]
print('apres 200 pas :', 'diverge -- aucune forme normale atteinte' if verdict is None else cb.to_str(verdict))

terme initial : S I I (S I I)
pas 0: S I I (S I I)
pas 1: I (S I I) (I (S I I))
pas 2: S I I (I (S I I))
pas 3: I (I (S I I)) (I (I (S I I)))
apres 200 pas : diverge -- aucune forme normale atteinte


**Lecture du resultat** : la trace grossit a chaque pas -- le terme se replie sur sa propre
structure sans jamais s'aplatir. C'est le materiau brut de l'indecidabilite : Church et Kleene
introduisent la lambda-definissabilite vers 1934, Turing les machines en 1936, et "that -- in
effect by thinking about having it compile itself -- Hilbert's Entscheidungsproblem couldn't be
solved" (*A Centennial View*, section *Godel's Theorem and Computability*). Le meme essay
rapporte que Church demontre l'indecidabilite de l'equivalence des expressions lambda avant que
Turing n'etablisse la correspondance avec ses machines (section *Lambda Calculus*). Un moteur
assez riche pour tout calculer est assez riche pour ne rien garantir.

## 4. L'abstraction de bracket : eliminer les variables

Le pont entre le lambda-calcul de Church et les combinateurs est l'**abstraction de bracket** :
traduire `\x. M` en S/K/I par trois schemas -- `[x.x] = I`, `[x.c] = K c` (c sans x),
`[x.(M N)] = S [x.M] [x.N]`. Wolfram releve que Church, "on the very first page he defines
lambda, he's referencing Schonfinkel's combinator paper" (*A Centennial View*, section *Lambda
Calculus*) : les deux formalismes sont nes enchevetres.

L'organe `cb.lam` implemente la traduction **pure**, sans les optimisations de Curry (combinateurs
B, C...) : la croissance du terme traduit est justement l'objet de la mesure -- optimiser la
traduction, ce serait maquiller le prix de l'elimination des variables.

In [5]:
print('\\x.x           ->', cb.to_str(cb.lam('x', cb.Var('x'))))
print('\\x.c           ->', cb.to_str(cb.lam('x', cb.Var('c'))))
print('\\x.f x (g x)   ->', cb.to_str(cb.lam('x', cb.parse_ski('f x (g x)'))))


def taille(t):
    """Nombre de noeuds (Var ou App) du terme."""
    if isinstance(t, cb.Var):
        return 1
    return 1 + taille(t.func) + taille(t.arg)


for n in (1, 2, 3, 4, 5):
    print(f'church({n}) en S/K pur : {taille(cb.church(n)):4d} noeuds')

\x.x           -> I
\x.c           -> K c
\x.f x (g x)   -> S (S (K f) I) (S (K g) I)
church(1) en S/K pur :   19 noeuds
church(2) en S/K pur :   35 noeuds
church(3) en S/K pur :   51 noeuds
church(4) en S/K pur :   67 noeuds
church(5) en S/K pur :   83 noeuds


**Lecture du resultat** : `\x. f x (g x)` devient `S (S (K f) I) (S (K g) I)` -- chaque variable
eliminee laisse une trace de structure. La mesure des tailles est nette : `church(n)` coute
`16 n + 3` noeuds en traduction pure, une croissance **lineaire mais au facteur 16**. Le prix
de l'elimination des variables n'est pas un rhetoric : c'est un facteur mesurable, et les
optimisations de Curry (B, C...) sont precisement l'ingenierie qui l'a combattu historiquement.

## 5. Les entiers de Church : l'arithmetique sans nombres

Un entier n est la fonction qui applique n fois son argument : `church(n) = \f x. f^n x`.
L'addition, la multiplication et l'exponentiation sont alors des **compositions** : ajouter,
c'est enchainer ; multiplier, c'est iterer l'iteration ; elever a une puissance, c'est iterer
la multiplication. Le carnet construit ces combinateurs par abstraction de bracket puis les
execute -- l'arithmetique est reelle, pas recitee.

In [6]:
print('church(2) =', cb.to_str(cb.church(2)))
print('round-trip 0..7 :', [cb.church_to_int(cb.church(n)) for n in range(8)])

plus = cb.lam('m', cb.lam('n', cb.lam('f', cb.lam('x', cb.parse_ski('m f (n f x)')))))
mult = cb.lam('m', cb.lam('n', cb.lam('f', cb.lam('x', cb.parse_ski('n (m f) x')))))
exp = cb.lam('m', cb.lam('n', cb.parse_ski('n m')))

deux, trois = cb.church(2), cb.church(3)
somme = cb.App(cb.App(plus, deux), trois)
produit = cb.App(cb.App(mult, deux), trois)
puissance = cb.App(cb.App(exp, deux), trois)
print('2 + 3 =', cb.church_to_int(somme), '| 2 * 3 =', cb.church_to_int(produit), '| 2 ^ 3 =', cb.church_to_int(puissance))

church(2) = S (S (K S) (S (K K) I)) (S (S (K S) (S (K K) I)) (K I))
round-trip 0..7 : [0, 1, 2, 3, 4, 5, 6, 7]
2 + 3 = 5 | 2 * 3 = 6 | 2 ^ 3 = 8


**Lecture du resultat** : `5`, `6`, `8` -- calcules par `S` et `K` seuls, sans aucun entier
natif jusqu'a la conversion finale. C'est le sens operationnel de la these d'universalite :
non pas un slogan, mais le fait que trois lignes de composition remplacent les tables
d'operation. La lecture `n (m f)` pour la multiplication -- iterer `m` fois, puis iterer ce
resultat `n` fois -- est le moment exact ou l'exposant devient une iteration d'iterations.

## 6. Church-Rosser empirique : mille chemins, une seule forme normale

Le **theoreme de Church-Rosser** (confluence) etablit que si un terme a deux formes normales,
elles sont egales : l'ordre de reduction ne change pas le resultat, seulement le chemin. Ce
carnet n'en donne **pas une preuve** -- il en donne le **temoin empirique** : depuis un meme
terme, reduire un redex **aleatoire** a chaque pas, 30 fois, et compter les formes normales
distinctes.

In [7]:
rng = random.Random(42)  # graine fixee : le temoin est reproductible
term = cb.App(cb.App(cb.church(5), cb.Var('f')), cb.Var('x'))
formes, pas_vus = set(), []
for _ in range(30):
    nf, n_pas = cb.random_reduce(term, rng, 20000)
    formes.add(cb.to_str(nf))
    pas_vus.append(n_pas)
print('formes normales distinctes :', len(formes))
for f in sorted(formes):
    print('   ', f)
print('pas de reduction : min', min(pas_vus), '| max', max(pas_vus), '| median', sorted(pas_vus)[15])

formes normales distinctes : 1
    f (f (f (f (f x))))
pas de reduction : min 42 | max 42 | median 42


**Lecture du resultat -- et sa limite, dite**. Les 30 ordres convergent vers la meme forme
normale `f (f (f (f (f x))))` : c'est le temoin. Sur ce terme precis, le nombre de pas est meme
constant (42) : aucun redex du terme n'en duplique ni n'en detruit un autre -- une propriete de
ce terme, pas de la reduction en general (le redex d'OMEGA, lui, se duplique a chaque pas).

**Un temoin n'est pas une preuve** : trente ordres ne couvrent pas l'infinite des chemins. La
preuve formelle existe desormais : le lake
[`combinators_lean`](../../SymbolicAI/Lean/combinators_lean/) (pli 4, grain B) demontre
`confluence` -- toute paire de reductions `step*` issues d'un meme terme se rejoint -- et son
corollaire `unique_normal_form` : deux reductions exhaustives d'un meme terme aboutissent au
meme terme irreductible. La preuve invoque l'organe Mathlib `Relation.church_rosser` (forme de
Hindley-Rosen : le losange local de la reduction parallele entraine la confluence de sa cloture
reflexive-transitive), menee de style Tait-Martin-Lof sans lieurs -- ce que S/K/I rend bien plus
simple que le lambda-calcul. Ce carnet fournit l'instrument de mesure ; le lake fournit la
preuve verifiee par le noyau Lean.

## 7. Le probleme de tag de Post : la borne d'irreductibilite

En 1920-1921, Emil Post simplifie la logique de Principia Mathematica en transformations de
chaines, jusqu'a un systeme minimal : alphabet `{0, 1}`, a chaque pas **supprimer 3 elements
en tete** puis **ajouter en queue** le bloc associe au premier element supprime (`0 -> 00`,
`1 -> 1101`). La question : **toute graine mene-t-elle a l'arret** (chaine plus courte que 3),
a un cycle, ou a une croissance sans borne ?

Wolfram raconte que Post "began on it in the early summer of 1921, no doubt expecting that such
a simple-to-state problem would have a correspondingly simple solution" (section *[Despite]
Considerable Effort... [It Proved] Intractable*) -- et que lui-meme doit reporter, un siecle
plus tard : "even a century later I still haven't been able to answer this question" (section
*So Does It Always Halt?*).

In [8]:
for graine in ('10', '1001010010000', '110101'):
    trace, verdict = cb.tag_evolve(graine, max_steps=100000)
    longueurs = [len(s) for s in trace]
    print(f'graine {graine!r:18} -> {verdict:9} | etats: {len(trace):5d} | longueurs min/max: {min(longueurs)}/{max(longueurs)}')
    print('   tete :', ["".join(s) for s in trace[:4]])
    print('   fin  :', ["".join(s) for s in trace[-2:]])

trace_long, verdict_long = cb.tag_evolve('1001001001001001001001001001001', max_steps=5000)
print('graine periodique (31 elements) ->', verdict_long,
      '| longueur max atteinte :', max(len(s) for s in trace_long))

graine '10'               -> halted    | etats:     1 | longueurs min/max: 2/2
   tete : ['10']
   fin  : ['10']
graine '1001010010000'    -> cycle     | etats:    33 | longueurs min/max: 8/16
   tete : ['1001010010000', '10100100001101', '001000011011101', '00001101110100']
   fin  : ['0000110111011101', '011011101110100']
graine '110101'           -> cycle     | etats:    22 | longueurs min/max: 6/16
   tete : ['110101', '1011101', '11011101', '111011101']
   fin  : ['0000110111011101', '011011101110100']
graine periodique (31 elements) -> cycle | longueur max atteinte : 208


**Lecture du resultat** : trois destins mesures -- **arret** immediat (`10` est plus courte que
le seuil), **cycle** (la graine de 13 elements de l'essai de Wolfram, et une graine de 6,
rejoignent le meme attracteur), **cycle apres croissance** (une graine periodique de 31 monte
jusqu'a 208 elements avant de retomber). Aucune graine testee ici ne croit sans borne -- mais
l'echantillon ne prouve rien : c'est exactement le statut du probleme.

Post lui-meme voyait plus loin que son echec : "I would have discovered Godel's theorem in
1921 -- if I had been Godel" (cite par Wolfram, meme section introductive). Et en 1961, Marvin
Minsky demontre qu'un systeme de tag **convenablement construit** peut calculer tout ce qu'une
machine de Turing calcule (section *[Despite] Considerable Effort...*) : le probleme de Post
n'est pas dur parce que le systeme est pauvre -- il est dur parce qu'il est deja riche.


## 8. Blocs inatteignables et r = 1 : ou passe la frontiere causale

Deux faits de structure, tous deux mesures par Wolfram (2021) :

1. **Le bloc `1111` est inatteignable** : "no sequence of the 00s and 1101s inserted by the rule
   can ever produce it" (section *The Basic Setup*). Une preuve de deux lignes sur les regles --
   mais une contrainte permanente sur tout l'espace d'etats.
2. **La suppression d'un seul element par pas (r = 1) tue la causalite** : "rules which delete
   only one element at each step cannot involve real communication (or causal connections)
   between different parts of the string, and must be equivalent to neighbor-independent
   substitution systems" -- les regles `0 -> 01, 1 -> 10` engendrent le mot de Thue-Morse,
   `0 -> 01, 1 -> 0` le mot de Fibonacci (section *Other Tag Systems*).

In [9]:
trace, verdict = cb.tag_evolve('1001010010000')
manquants = cb.first_missing_blocks(trace, 4)
print('blocs de largeur 4 absents de la trace :', manquants)
print('1111 absent :', '1111' in manquants)


def substitue(s, regles):
    """Une generation d'un systeme de substitution."""
    return ''.join(regles[c] for c in s)


thue = '0'
for _ in range(6):
    thue = substitue(thue, {'0': '01', '1': '10'})
fibo = '0'
for _ in range(7):
    fibo = substitue(fibo, {'0': '01', '1': '0'})
print('Thue-Morse (%d) :' % len(thue), thue)
print('Fibonacci (%d)  :' % len(fibo), fibo)

# Les longueurs affichees sont mesurees, pas recopiees : un changement du nombre de
# generations ne peut plus desynchroniser l'etiquette du mot qu'elle annonce.
# Un etat de 21 elements ne se lit pas forcement dans le prefixe affiche (64) : les
# facteurs se mesurent sur un prefixe plus long que la fenetre d'affichage.
thue_long = thue
for _ in range(4):
    thue_long = substitue(thue_long, {'0': '01', '1': '10'})
fibo_long = fibo
for _ in range(6):
    fibo_long = substitue(fibo_long, {'0': '01', '1': '0'})


def evolue_r1(graine, regles, n_pas):
    """Systeme de tag a r = 1 : chaque element, a son tour, est remplace par son bloc."""
    etat, etats = list(graine), [''.join(graine)]
    for _ in range(n_pas):
        tete = etat.pop(0)
        etat.extend(regles[tete])
        etats.append(''.join(etat))
    return etats


etats_tm = evolue_r1('0', {'0': '01', '1': '10'}, 20)
etats_fibo = evolue_r1('0', {'0': '01', '1': '0'}, 20)
print('sur %d etats r=1 :' % len(etats_tm))
print('  prefixes de Thue-Morse :', sum(1 for e in etats_tm if thue_long.startswith(e)))
print('  facteurs de Thue-Morse :', sum(1 for e in etats_tm if e in thue_long))
print('  prefixes de Fibonacci  :', sum(1 for e in etats_fibo if fibo_long.startswith(e)))
print('  facteurs de Fibonacci  :', sum(1 for e in etats_fibo if e in fibo_long))


blocs de largeur 4 absents de la trace : ['1100', '1111']
1111 absent : True
Thue-Morse (64) : 0110100110010110100101100110100110010110011010010110100110010110
Fibonacci (34)  : 0100101001001010010100100101001001
sur 21 etats r=1 :
  prefixes de Thue-Morse : 5
  facteurs de Thue-Morse : 21
  prefixes de Fibonacci  : 6
  facteurs de Fibonacci  : 21


**Lecture du resultat** : a `r = 1`, les etats visites sont des **facteurs** (sous-mots) des
mots de Thue-Morse et de Fibonacci -- des suites parfaitement regulieres, generees par
substitution, sans aucune communication entre parties de la chaine. Ils ne sont pas
**prefixes** pour autant : 5 des 21 etats seulement sont des prefixes du mot de Thue-Morse, 6
pour Fibonacci ; les autres demarrent plus loin dans le mot. Le systeme de Post, lui, supprime
**trois** elements par pas : ce que la tete supprime depend de ce que la queue a recu, la
boucle causale est fermee, et la trace y laisse des trous -- `1111` est absent de la trace de
la graine de reference, et aucun bloc ajoute (`00` ou `1101`) ne peut le fabriquer, faute de
quatre `1` consecutifs ou de deux `1` terminaux. Une graine arbitraire, elle, en produit : les
496 graines de longueur 4 a 8 sont 121 a en faire apparaitre, et c'est l'exercice 3 qui le
mesure. La frontiere entre le systeme de substitution (compris) et le tag (ouvert) passe la --
entre `r = 1` et `r = 2`, un seul element supprime de plus.


## 9. Reception critique : un siecle de combinateurs, lue depuis 2021

La these wolframienne -- Schonfinkel fournit "the first explicit example of a system capable of
what we now call universal computation" (*S Combinator Challenge*, section *Hiding in Plain
Sight for a Century?*) -- se confronte a une tradition que Wolfram lui-meme a patiemment
relevee : *A Bibliography of Combinators* (arXiv 2106.11729) recense **plus de 200 entrees**
sur un siecle, de Curry (a partir de 1927-1930) aux travaux modernes de term rewriting.

| Volet de la reception | Reference | Ce qu'elle oppose a la these |
|---|---|---|
| Systematisation | Curry, Hindley & Seldin, *Combinatory Logic* vol. II (1972) | le programme n'est pas une decouverte isolee de 1920 : c'est quarante ans de construction de Curry, achevee en 1972 |
| Synthese moderne | Hindley & Seldin, *Lambda-Calculus and Combinators: an Introduction* (1980/2008) | le lambda-calcul et les combinateurs forment un seul corpus enseignable, pas un curiosity isole |
| Histoire | Cardone & Hindley, *History of Lambda-calculus and Combinatory Logic* (2006/2009) | les combinateurs se comprennent dans la continuite Frege-Principia-Church, pas comme un big bang |
| Acces canonique | Bimbo, *Combinatory Logic*, Stanford Encyclopedia of Philosophy (2009) | la communaute logique a son article de reference -- la place des combinateurs est institutionnalisee |
| Comportement computationnel | Klop, van Oostrom & van Raamsdonk, *Combinatory term rewriting systems* (TCS 121, 1993) | la theorie du rewriting a traite les combinateurs comme objet d'etude a part entiere, bien avant la ruliologie |

Ces references sont relevees dans la **bibliographie archivee de Wolfram lui-meme** (PDF
arXiv 2106.11729) -- le carnet cite la grille de lecture wolframienne et ses propres sources,
pas les originaux : c'est une reception de reception, et il le dit.

Le point de friction le plus net : la these de la **representation ultime**. Wolfram ecrit les
combinateurs comme le fondement le plus simple du calcul ; la tradition Curry-Hindley les traite
comme un **formalisme commode** -- une notation sans variables pour raisonner sur les fonctions
et les modeles de la logique combinatoire. Les deux lectures ne se contredisent pas ; elles ne
parlent pas de la meme question. Le S Combinator Challenge ($20 000, 2021) est la seule piece
qui les force a se rejoindre : si S seul est universel, la these du minimum absolu se renforce
de l'interieur du formalisme -- et le challenge est, a ce jour, non resolu.

## 10. Synthese -- trois statuts, un seul instrument

| Affirmation | Source | Statut | Ce que ce carnet a fait |
|---|---|---|---|
| S K K x -> x : I est derivable | Schonfinkel 1924 | theoreme trivial | reduit pas a pas (section 1) |
| L'ordre de reduction change l'issu : K I OMEGA | standardisation | theoreme | les deux ordres executes (section 2) |
| S/K represente l'arithmetique | Church 1936 | theoreme | 2+3, 2*3, 2^3 calcules (section 5) |
| Toute forme normale est unique | Church-Rosser | theoreme | **temoin empirique** : 30 ordres -> 1 forme (section 6) ; preuve formelle = grain B (Lean) |
| 1111 est inatteignable dans le tag de Post | Wolfram 2021, *The Basic Setup* | fait prouvable sur les regles | mesure sur la trace (section 8) |
| Le tag de Post s'arrete-t-il toujours ? | Post 1921 | **probleme ouvert** | comportements mesures, aucun tranche (section 7) |
| S seul est-il universel ? | Challenge 2021 | probleme ouvert ($20 000) | cadre expose, non attaque (section 9) |

Le pli 4 du Origami Wolfram demandait la reception critique "au meme rang que les theses" :
c'est la table ci-dessus -- chaque these a son statut, chaque statut a sa charge de preuve, et
le carnet ne depasse jamais la sienne.

## Vérifiez votre compréhension

1. La section 2 exécute les deux ordres de réduction sur `K I OMEGA` — quel ordre termine, et que montre celui qui ne termine pas ?
2. L'arithmétique de Church est calculée en section 5 — pourquoi `2^3` n'est-il pas symétrique à `3^2` ?
3. Church-Rosser est attesté en section 6 par 30 ordres vers une forme normale — pourquoi cela reste-t-il un témoin empirique et non une preuve ?

## Exercice 1 -- Le combinateur d'echange, en pur S/K

Le combinateur `C` echange ses deux derniers arguments : `C f x y -> f y x`. Construisez-le
par abstraction de bracket (`cb.lam`, trois variables liees), appliquez-le a `f x y` et
verifiez que la forme normale est `f y (x)` -- c'est-a-dire `f y x` en association a gauche.

**Etapes** : (1) ecrire `C` comme `\f x y. f y x` via trois `cb.lam` imbriques ; (2) construire
`C f x y` avec `cb.App` ; (3) reduire en ordre normal et afficher la forme normale ; (4) compter
les noeuds de `C` avec la fonction `taille` de la section 4 -- le prix de l'echange.

In [10]:
# TODO etudiant : construire C = \f x y. f y x par abstraction de bracket
C = None  # Etape 1 : cb.lam('f', cb.lam('x', cb.lam('y', ...)))

# Etape 2 : appliquer C a f, x, y
applique = None  # TODO etudiant

# Etapes 3-4 : reduire et mesurer
if C is not None and applique is not None:
    nf, steps, _ = cb.reduce_term(applique, 'normal', 500)
    print('forme normale :', cb.to_str(nf), f'({steps} pas)')
else:
    print('Exercice a completer : C et applique')

Exercice a completer : C et applique


## Exercice 2 -- L'exponentiation n'est pas symetrique

La section 5 calcule `2^3 = 8` avec `exp = \m n. n m`. Verifiez que **l'ordre des operandes
compte** : `3^2` doit donner `9`, pas `8`. Construisez les deux termes, reduisez-les et
convertissez avec `cb.church_to_int`.

**Etapes** : (1) reconstruire `exp` par `cb.lam` ; (2) `exp trois deux` et `exp deux trois` ;
(3) convertir les deux resultats ; (4) expliquer en une phrase pourquoi `n m` donne `m^n`
et non `m^n` inverse -- que represente l'application de `n` a `m` ?

In [11]:
deux, trois = cb.church(2), cb.church(3)
exp = None  # TODO etudiant : \m n. n m par cb.lam

if exp is not None:
    trente_deux = cb.App(cb.App(exp, trois), deux)   # 3^2 attendu : 9
    huit = cb.App(cb.App(exp, deux), trois)           # 2^3 attendu : 8
    print('3 ^ 2 =', cb.church_to_int(trente_deux), '| 2 ^ 3 =', cb.church_to_int(huit))
else:
    print('Exercice a completer : exp')

Exercice a completer : exp


## Exercice 3 -- Cartographier les graines du tag de Post

La section 7 teste quatre graines. Ecrivez un balayage : pour toutes les graines binaires de
longueur 4 a 8, executez `cb.tag_evolve` (borne `max_steps=100000`), comptez les verdicts
`halted` / `cycle` / `unbounded`, et relevez la plus longue croissance avant cycle.

**Etapes** : (1) generer les graines avec `itertools.product('01', repeat=L)` ; (2) classer
chaque verdict ; (3) afficher le decompte par longueur ; (4) relever la graine dont le
transitoire est le plus long -- la reponse reste nettement sous 100 etats, verifiez-le ;
(5) montrer que le transitoire peut pourtant depasser 100 etats, mais hors de ce domaine :
cherchez une graine de longueur superieure a 8 (il en existe une des la longueur 12), et
verifiez que votre mesure est reproductible (le systeme est deterministe : deux executions
donnent le meme verdict).


In [12]:
import itertools

decompte = None  # TODO etudiant : {(L, verdict): nombre}
transitoire_max = None  # TODO etudiant : (graine, nombre d'etats avant cycle)

print('Exercice a completer : cartographier les graines de longueur 4 a 8')
print('Indice : cb.tag_evolve(graine) renvoie (trace, verdict) ; len(trace) mesure le transitoire.')

Exercice a completer : cartographier les graines de longueur 4 a 8
Indice : cb.tag_evolve(graine) renvoie (trace, verdict) ; len(trace) mesure le transitoire.


## À retenir

*Ce carnet couvre le premier point du pli 4 : les combinateurs S/K/I comme representation
ultime du calcul -- syntaxe, reduction, strategies, arithmetique de Church -- et la borne
d'irreductibilite que le systeme de tag de Post leur oppose depuis 1921. Trois statuts y
coexistent sans se confondre : des theoremes executes (`S K K x -> x`, l'arithmetique), un
temoin empirique d'un theoreme non prouve ici (Church-Rosser), et deux problemes ouverts
(l'arret du tag de Post, l'universalite de S seul).*

Ce qui reste au pli : la **preuve formelle de la confluence** (grain B, organe Lean -- les
relations de Mathlib y seront nommees avant toute implementation), et la lecture des textes
2025 (*Ruliology of Lambdas*) que la matrice du pli signale sans les avoir encore extraits.
La lecon de methode vaut pour toute la serie : un moteur minimal n'est pas un moteur pauvre --
S et K tiennent en deux lignes, et un siecle de reception n'a pas epuise ce qu'elles contiennent.

## Pour aller plus loin

- **[ICT-Synthèse — CrossSubstrat](ICT-Synthese-CrossSubstrat-Python.ipynb)** : le capstone de la série, qui mesure ce que les instruments transfèrent d'un substrat à l'autre.
- **[ICT-0 — Cadrage de la série](ICT-0-Framing.md)** : le principe directeur de la série ICT.